# Análise de Dados de Wearables — PAMAP2

**Objetivo:** Ganhar proficiência com bibliotecas de Data Science (pandas, scikit-learn, seaborn) aplicadas a um problema real de classificação de atividades físicas.

O pipeline cobre três etapas principais:
1. Carregamento e limpeza dos dados brutos do dataset PAMAP2
2. Análise exploratória
3. Três tarefas de ML/análise: **classificação** (caminhada/corrida/descanso), **detecção de intensidade** e **estimativa de gasto calórico**

## 1. Imports e configuração

Todas as dependências do projeto em um único lugar para facilitar a instalação e leitura.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

## 2. Estrutura do dataset

O PAMAP2 contém leituras de três sensores IMU (mão, peito e calcanhar) de 9 indivíduos realizando até 18 atividades diferentes. Aqui definimos os metadados: mapeamento de IDs de atividade, caminhos dos arquivos e nomes das colunas.

In [ ]:
id_atividades = {
    0: 'transient',       1: 'lying',            2: 'sitting',
    3: 'standing',        4: 'walking',           5: 'running',
    6: 'cycling',         7: 'Nordic_walking',    9: 'watching_TV',
    10: 'computer_work',  11: 'car driving',      12: 'ascending_stairs',
    13: 'descending_stairs', 16: 'vacuum_cleaning', 17: 'ironing',
    18: 'folding_laundry', 19: 'house_cleaning',  20: 'playing_soccer',
    24: 'rope_jumping'
}

lista_dados = [
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject101.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject102.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject103.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject104.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject105.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject106.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject107.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject108.dat',
    '/home/davi/Documents/Projetos/Wearables/PAMAP2_Dataset/Protocol/subject109.dat',
]

col_geral          = ['timestamp', 'id_atividade', 'Frequência cardiaca (bpm)']

col_dados_mao      = ['mTemperatura',
                      'mAcel 16g (ms) 1', 'mAcel 16g (ms) 2', 'mAcel 16g (ms) 3',
                      'mAcel 6g (ms) 1',  'mAcel 6g (ms) 2',  'mAcel 6g (ms) 3',
                      'mGyro (rad/s) 1',  'mGyro (rad/s) 2',  'mGyro (rad/s) 3',
                      'mMag (μT) 1',      'mMag (μT) 2',      'mMag (μT) 3',
                      'mOrientação 1',    'mOrientação 2',    'mOrientação 3', 'mOrientação 4']

col_dados_peito    = ['pTemperatura',
                      'pAcel 16g (ms) 1', 'pAcel 16g (ms) 2', 'pAcel 16g (ms) 3',
                      'pAcel 6g (ms) 1',  'pAcel 6g (ms) 2',  'pAcel 6g (ms) 3',
                      'pGyro (rad/s) 1',  'pGyro (rad/s) 2',  'pGyro (rad/s) 3',
                      'pMag (μT) 1',      'pMag (μT) 2',      'pMag (μT) 3',
                      'pOrientação 1',    'pOrientação 2',    'pOrientação 3', 'pOrientação 4']

col_dados_calcanhar = ['cTemperatura',
                       'cAcel 16g (ms) 1', 'cAcel 16g (ms) 2', 'cAcel 16g (ms) 3',
                       'cAcel 6g (ms) 1',  'cAcel 6g (ms) 2',  'cAcel 6g (ms) 3',
                       'cGyro (rad/s) 1',  'cGyro (rad/s) 2',  'cGyro (rad/s) 3',
                       'cMag (μT) 1',      'cMag (μT) 2',      'cMag (μT) 3',
                       'cOrientação 1',    'cOrientação 2',    'cOrientação 3', 'cOrientação 4']

colunas = col_geral + col_dados_mao + col_dados_peito + col_dados_calcanhar

## 3. Carregamento e limpeza dos dados

Os arquivos `.dat` são lidos e concatenados em um único DataFrame. Em seguida, `LimpaDados` remove as colunas de orientação quaternion (ruidosas e pouco úteis para este projeto), descarta amostras de transição (`id_atividade == 0`) e preenche valores ausentes por interpolação linear — adequado para séries temporais de sensores.

In [ ]:
dados_completos = pd.DataFrame()

for arq in lista_dados:
    proc_dados = pd.read_table(filepath_or_buffer=arq, header=None, sep=r'\s+')
    proc_dados.columns = colunas
    dados_completos = pd.concat([dados_completos, proc_dados], ignore_index=True)


def LimpaDados(df):
    colunas_orientacao = [
        'cOrientação 1', 'cOrientação 2', 'cOrientação 3', 'cOrientação 4',
        'mOrientação 1', 'mOrientação 2', 'mOrientação 3', 'mOrientação 4',
        'pOrientação 1', 'pOrientação 2', 'pOrientação 3', 'pOrientação 4',
    ]
    df = df.drop(columns=colunas_orientacao)
    df = df.drop(df[df.id_atividade == 0].index)   # remove amostras de transição
    df = df.apply(lambda x: pd.to_numeric(x, errors='coerce'))
    df = df.interpolate()                           # preenche NaN por interpolação linear
    return df


dn = LimpaDados(dados_completos)
dn.reset_index(drop=True, inplace=True)
dn.head(10)

## 4. Divisão treino / teste

Separamos 80% dos dados para treino e 20% para teste usando amostragem aleatória. O `random_state` garante reprodutibilidade.

> **Nota:** esta abordagem pode vazar informações entre sujeitos (dados do mesmo indivíduo aparecem nos dois conjuntos). Para uma avaliação mais rigorosa, considere separar por `id_individuo` no futuro.

In [ ]:
treino_Dn = dn.sample(frac=0.8, random_state=1)
teste_Dn  = dn.drop(treino_Dn.index)

print(f'Amostras de treino : {len(treino_Dn):,}')
print(f'Amostras de teste  : {len(teste_Dn):,}')

## 5. Análise exploratória — Frequência Cardíaca

O boxplot da frequência cardíaca nos dá uma visão rápida da distribuição geral: mediana, dispersão e possíveis outliers. Isso ajuda a calibrar os limiares usados na detecção de intensidade mais adiante.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
plt.title('FC (BPM)')
sns.boxplot(y=treino_Dn['Frequência cardiaca (bpm)'], ax=ax)
plt.tight_layout()
plt.show()

## 6. Classificação: caminhada / corrida / descanso

Filtramos apenas as três classes de interesse e criamos uma feature de **magnitude do acelerômetro** para cada sensor — ela é invariante à orientação do dispositivo no corpo, o que a torna mais robusta que os eixos individuais.

O modelo escolhido é o **Random Forest**: lida bem com dados ruidosos de sensores, não exige normalização e fornece importância de features nativamente. O `classification_report` ao final mostra precisão, recall e F1 por classe.

In [ ]:
mapa_atividades = {1: 'descanso', 2: 'descanso', 3: 'descanso',
                   4: 'caminhada', 5: 'corrida'}

df_filtrado = dn[dn['id_atividade'].isin(mapa_atividades.keys())].copy()
df_filtrado['rotulo'] = df_filtrado['id_atividade'].map(mapa_atividades)

# Magnitude euclidiana do acelerômetro de 16g para cada sensor
for prefixo in ['m', 'p', 'c']:
    df_filtrado[f'{prefixo}Magnitude'] = np.sqrt(
        df_filtrado[f'{prefixo}Acel 16g (ms) 1']**2 +
        df_filtrado[f'{prefixo}Acel 16g (ms) 2']**2 +
        df_filtrado[f'{prefixo}Acel 16g (ms) 3']**2
    )

caracteristicas = ['Frequência cardiaca (bpm)', 'mMagnitude', 'pMagnitude', 'cMagnitude']

X = df_filtrado[caracteristicas]
y = df_filtrado['rotulo']

X_treino = X.loc[treino_Dn.index.intersection(X.index)]
y_treino = y.loc[X_treino.index]
X_teste  = X.loc[teste_Dn.index.intersection(X.index)]
y_teste  = y.loc[X_teste.index]

classificador = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
classificador.fit(X_treino, y_treino)

print(classification_report(y_teste, classificador.predict(X_teste)))

## 7. Detecção de intensidade do exercício

Classificamos cada amostra em três zonas de intensidade com base na frequência cardíaca — abordagem padrão da fisiologia do exercício:

| Zona | FC (bpm) |
|------|----------|
| Baixa | < 100 |
| Moderada | 100 – 139 |
| Alta | ≥ 140 |

O gráfico de barras empilhadas mostra como cada atividade distribui suas amostras entre as zonas, revelando o perfil de esforço típico de cada uma.

In [ ]:
def classificar_intensidade(bpm):
    if bpm < 100:
        return 'baixa'
    elif bpm < 140:
        return 'moderada'
    else:
        return 'alta'

dn['intensidade'] = dn['Frequência cardiaca (bpm)'].apply(classificar_intensidade)

fig, eixo = plt.subplots(figsize=(12, 5))
contagem_intensidade = (
    dn.groupby(['id_atividade', 'intensidade'])
      .size()
      .unstack(fill_value=0)
)
contagem_intensidade.plot(kind='bar', stacked=True, ax=eixo, colormap='RdYlGn_r')
eixo.set_xlabel('ID da Atividade')
eixo.set_ylabel('Número de amostras')
eixo.set_title('Distribuição de Intensidade por Atividade')
plt.tight_layout()
plt.show()

## 8. Estimativa de gasto calórico

Usamos o método **MET (Metabolic Equivalent of Task)**, padrão na literatura de wearables quando não há medição direta de VO₂.

A fórmula é:

$$\text{kcal} = \text{MET} \times \text{peso}_{kg} \times \text{tempo}_{horas}$$

Como o PAMAP2 é amostrado a **100 Hz**, cada linha representa 1/100 s = 1/360.000 horas. O cálculo é feito de forma **vetorizada** com `.map()`, que é muito mais eficiente que percorrer linha a linha com `apply`.

In [ ]:
valores_met = {
    1: 0.9,    # deitado
    2: 1.0,    # sentado
    3: 1.2,    # em pé
    4: 3.5,    # caminhada
    5: 8.0,    # corrida
    6: 6.0,    # ciclismo
    7: 4.5,    # caminhada nórdica
    12: 4.0,   # subindo escadas
    13: 3.0,   # descendo escadas
    16: 3.5,   # aspirando o pó
    17: 2.3,   # passando roupa
    20: 7.0,   # futebol
    24: 10.0,  # pular corda
}

PESO_CORPORAL_KG  = 70
TAXA_AMOSTRAGEM   = 100          # Hz
HORAS_POR_AMOSTRA = 1 / TAXA_AMOSTRAGEM / 3600

# Cálculo vetorizado: muito mais rápido que apply() linha a linha
dn['id_atividade'] = dn['id_atividade'].fillna(0).astype(int)
dn['met']          = dn['id_atividade'].map(valores_met).fillna(1.0)
dn['calorias_kcal'] = dn['met'] * PESO_CORPORAL_KG * HORAS_POR_AMOSTRA
dn.drop(columns=['met'], inplace=True)

resumo_calorias = dn.groupby('id_atividade')['calorias_kcal'].sum().reset_index()
resumo_calorias.columns = ['ID Atividade', 'Total kcal']
resumo_calorias['Atividade'] = resumo_calorias['ID Atividade'].map(id_atividades)

resumo_calorias.sort_values('Total kcal', ascending=False).reset_index(drop=True)